In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

In [0]:
trip_df = spark.table("silver_trips") 
rider_df = spark.table("silver_riders") 
station_df = spark.table("silver_stations")
payment_df = spark.table("silver_payments")

In [0]:
# CREATE DATE DIMENSION

date_df = trip_df.select(
    col("started_at").alias("full_date")
).union(
    trip_df.select(
        col("ended_at").alias("full_date")
    )
).union(
    payment_df.select(
        col("payment_date").cast("timestamp").alias("full_date")
    )
).distinct()

dim_date = date_df.withColumn(
    "date_key",
    monotonically_increasing_id()
)

dim_date = dim_date.withColumn(
    "day",
    dayofmonth("full_date")
).withColumn(
    "month",
    month("full_date")
).withColumn(
    "quarter",
    quarter("full_date")
).withColumn(
    "year",
    year("full_date")
).withColumn(
    "day_of_week",
    date_format("full_date", "E")
)

dim_date.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_dim_date")

In [0]:
dim_riders = rider_df.select(
    "rider_id",
    "first_name",
    "last_name",
    "address",
    "birthday",
    "account_start_date",
    "is_member"
)

dim_riders.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_dim_riders")

In [0]:
dim_stations = station_df.select(
    "station_id",
    "station_name",
    "latitude",
    "longitude"
)

dim_stations.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_dim_stations")

In [0]:
fact_payments = payment_df.select(
    "payment_id",
    "payment_date",
    "amount",
    "rider_id"
)

fact_payments.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_fact_payments")

In [0]:
spark.sql("DROP TABLE IF EXISTS gold_fact_trips")

DataFrame[]

In [0]:
fact_trips = trip_df.join(
    rider_df,
    trip_df.rider_id == rider_df.rider_id,
    "left"
)

fact_trips = fact_trips.withColumn(
    "trip_duration_minutes",
    (
        unix_timestamp("ended_at") -
        unix_timestamp("started_at")
    ) / 60
)

fact_trips = fact_trips.withColumn(
    "rider_age_at_trip",
    floor(
        datediff(
            col("started_at"),
            col("birthday")
        ) / 365
    )
)

fact_trips = fact_trips.select(
    trip_df["ride_id"],
    trip_df["rideable_type"],
    trip_df["started_at"],
    trip_df["ended_at"],
    trip_df["start_station_id"],
    trip_df["end_station_id"],
    trip_df["rider_id"],
    "trip_duration_minutes",
    "rider_age_at_trip"
)

fact_trips.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_fact_trips")


In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

+--------+------------------+-----------+
|database|tableName         |isTemporary|
+--------+------------------+-----------+
|default |bronze_payments   |false      |
|default |bronze_riders     |false      |
|default |bronze_stations   |false      |
|default |bronze_trips      |false      |
|default |gold_dim_date     |false      |
|default |gold_dim_payments |false      |
|default |gold_dim_riders   |false      |
|default |gold_dim_stations |false      |
|default |gold_fact_payments|false      |
|default |gold_fact_trips   |false      |
|default |payments          |false      |
|default |riders            |false      |
|default |silver_payments   |false      |
|default |silver_riders     |false      |
|default |silver_stations   |false      |
|default |silver_trips      |false      |
|default |stations          |false      |
|default |trips             |false      |
+--------+------------------+-----------+

